<a href="https://colab.research.google.com/github/MINNNN0119/411312456_-B/blob/main/411312456_%E6%9C%9F%E4%B8%AD%E8%80%83B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install groq

In [22]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（老師提供，請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


一.規格書填寫內容學號：41312456    姓名：陳彥閔    卷別：B   1. 系統目的（一句話）建立一個整合 Groq AI 語意解析、SQLite 資料庫與 Gradio 介面的校園咖啡廳智慧預約取餐與銷售統計系統。

二.使用者會怎麼說

1.「幫我訂兩杯美式咖啡」

2.「我要取消剛剛的拿鐵訂單」

3.「幫我查詢抹茶歐蕾還剩幾杯」

4.「今天各品項過得怎樣」

5.「我要訂一杯拿鐵加抹茶歐蕾」

| 表名 | 欄位 | 型態 | 說明 |
| :--- | :--- | :--- | :--- |
| `menu` | `item` | TEXT (PK) | 品項名稱（美式咖啡、拿鐵、抹茶歐蕾） |
| `menu` | `price` | INTEGER | 單價（元） |
| `menu` | `stock` | INTEGER | 今日庫存（杯） |
| `orders` | `order_id` | INTEGER (PK) | 訂單編號（自動遞增） |
| `orders` | `name` | TEXT | 訂購人姓名 |
| `orders` | `item` | TEXT | 訂購品項 |
| `orders` | `quantity` | INTEGER | 訂購數量 |
| `orders` | `total_price` | INTEGER | 總金額（單價 × 數量） |
| `orders` | `status` | TEXT | 訂單狀態（「已成立」或「已取消」） |

In [15]:
import sqlite3
import pandas as pd
from IPython.display import display

conn = sqlite3.connect("coffee_shop.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS orders")
cursor.execute("DROP TABLE IF EXISTS menu")

cursor.execute("""
CREATE TABLE menu (
    item TEXT PRIMARY KEY,
    price INTEGER,
    stock INTEGER
)
""")

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    item TEXT,
    quantity INTEGER,
    total_price INTEGER,
    status TEXT
)
""")

initial_menu = [
    ("美式咖啡", 50, 5),
    ("拿鐵", 65, 4),
    ("抹茶歐蕾", 70, 3)
]
cursor.executemany("INSERT INTO menu (item, price, stock) VALUES (?, ?, ?)", initial_menu)
conn.commit()

df_menu = pd.read_sql("SELECT * FROM menu", conn)
display(df_menu)

,item,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3


In [24]:
def ai_parse_intent(user_input):
    prompt = f"""
你是一個咖啡廳系統的語意解析助理。
請從使用者的話語中萃取出三個欄位：動作、品項、數量。
- 動作只能是：「訂購」、「取消」、「查詢」、「統計」。
- 品項只能是：「美式咖啡」、「拿鐵」、「抹茶歐蕾」。
- 數量：請提取數字。若沒說數量，預設視為 1。
- 規則：
  1. 如果使用者問到「今天各品項過得怎樣」、「賣得怎樣」等統計相關問題，動作請輸出「統計」，品項輸出「無法判斷」，數量輸出「1」。
  2. 如果品項超出範圍（例如珍珠奶茶），品項請輸出「無法判斷」。
  3. 如果完全無法判斷，請輸出：無法判斷,無法判斷,0

請嚴格僅輸出以下格式（不要有其他多餘文字）：
動作,品項,數量

使用者說的話是：{user_input}
"""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        result = response.choices[0].message.content.strip()
        return result
    except Exception as e:
        return "無法判斷,無法判斷,0"

In [21]:
def process_order(name, user_input):
    ai_result = ai_parse_intent(user_input)
    parts = ai_result.split(",")
    if len(parts) != 3:
        return f"AI 解析錯誤，請重新表達您的需求。（解析結果：{ai_result}）"

    action, item, qty_str = parts[0].strip(), parts[1].strip(), parts[2].strip()

    try:
        qty = int(qty_str)
    except:
        qty = 1

    # 根據規則：訂購與取消需要姓名，查詢不需要
    if action in ["訂購", "取消"] and (not name or name.strip() == ""):
        return f"【提示】進行「{action}」時請先輸入您的姓名！"

    conn = sqlite3.connect("coffee_shop.db")
    cursor = conn.cursor()

    if action == "訂購" and (qty < 1 or qty > 3):
        conn.close()
        return f"【拒絕】數量必須在 1 到 3 之間（您輸入了 {qty} 杯），訂購失敗。"

    if action == "查詢":
        if item not in ["美式咖啡", "拿鐵", "抹茶歐蕾"]:
            conn.close()
            # 無法判斷品項時，回覆目前有哪些品項
            cursor.execute("SELECT item FROM menu")
            items = [row[0] for row in cursor.fetchall()]
            conn.close()
            return f"找不到該品項。目前提供的品項有：{', '.join(items)}"

        cursor.execute("SELECT stock FROM menu WHERE item = ?", (item,))
        row = cursor.fetchone()
        conn.close()
        return f"【查詢結果】{item} 今日剩餘庫存還有 {row[0]} 杯。"

    elif action == "統計":
        query = """
        SELECT
            m.item,
            m.price,
            m.stock AS remaining_stock,
            COALESCE(SUM(CASE WHEN o.status = '已成立' THEN o.quantity ELSE 0 END), 0) as sold_qty,
            (m.stock + COALESCE(SUM(CASE WHEN o.status = '已成立' THEN o.quantity ELSE 0 END), 0)) as initial_stock,
            (COALESCE(SUM(CASE WHEN o.status = '已成立' THEN o.quantity ELSE 0 END), 0) * m.price) as revenue
        FROM menu m
        LEFT JOIN orders o ON m.item = o.item
        GROUP BY m.item, m.price, m.stock
        """
        df_stats = pd.read_sql(query, conn)
        conn.close()

        output = "【今日銷售統計報表】\n"
        for _, row in df_stats.iterrows():
            output += f"- 品項：{row['item']} | 單價：{row['price']} 元 | 初始庫存：{row['initial_stock']} 杯 | 已售：{row['sold_qty']} 杯 | 剩餘庫存：{row['remaining_stock']} 杯 | 營收：{row['revenue']} 元\n"
        return output

    elif action == "訂購":
        if item not in ["美式咖啡", "拿鐵", "抹茶歐蕾"]:
            conn.close()
            cursor.execute("SELECT item FROM menu")
            items = [row[0] for row in cursor.fetchall()]
            conn.close()
            return f"AI 無法辨識品項。目前提供的品項有：{', '.join(items)}"

        cursor.execute("SELECT SUM(quantity) FROM orders WHERE name = ? AND status = '已成立'", (name,))
        res = cursor.fetchone()
        current_user_total = res[0] if res[0] is not None else 0

        if current_user_total + qty > 3:
            conn.close()
            return f"【訂購失敗】每位學生今天最多只能訂 3 杯（您目前已訂 {current_user_total} 杯）。"

        cursor.execute("SELECT price, stock FROM menu WHERE item = ?", (item,))
        menu_row = cursor.fetchone()
        if not menu_row:
            conn.close()
            return "品項不存在。"

        price, stock = menu_row[0], menu_row[1]

        if stock < qty:
            conn.close()
            return f"【訂購失敗】「{item}」庫存不足（剩餘 {stock} 杯）。"

        total_price = price * qty
        cursor.execute("UPDATE menu SET stock = stock - ? WHERE item = ?", (qty, item))
        cursor.execute("INSERT INTO orders (name, item, quantity, total_price, status) VALUES (?, ?, ?, ?, '已成立')",
                       (name, item, qty, total_price))
        conn.commit()
        conn.close()
        return f"【成功訂購】感謝 {name}！成功預訂 {qty} 杯「{item}」，總金額 {total_price} 元。"

    elif action == "取消":
        cursor.execute("SELECT order_id, item, quantity FROM orders WHERE name = ? AND status = '已成立' ORDER BY order_id DESC LIMIT 1", (name,))
        order_row = cursor.fetchone()

        if not order_row:
            conn.close()
            return f"【取消失敗】查無 {name} 的有效訂單可供取消。"

        o_id, o_item, o_qty = order_row[0], order_row[1], order_row[2]

        cursor.execute("UPDATE orders SET status = '已取消' WHERE order_id = ?", (o_id,))
        cursor.execute("UPDATE menu SET stock = stock + ? WHERE item = ?", (o_qty, o_item))
        conn.commit()
        conn.close()
        return f"【成功取消】{name} 的最新訂單（訂單編號 #{o_id}：{o_qty} 杯 {o_item}）已取消。"

    else:
        conn.close()
        cursor.execute("SELECT item FROM menu")
        items = [row[0] for row in cursor.fetchall()]
        return f"無法判斷您的需求。目前有的品項為：{', '.join(items)}"

In [20]:
import gradio as gr
import sqlite3
import pandas as pd

def get_menu_string():
    conn = sqlite3.connect("coffee_shop.db")
    df_menu = pd.read_sql("SELECT * FROM menu", conn)
    conn.close()
    return df_menu.to_string(index=False)

def chatbot_interface(name, message):
    if not message.strip():
        return "請輸入您的訊息。", get_menu_string()
    response = process_order(name, message)
    return response, get_menu_string()

with gr.Blocks(title="校園咖啡廳預約取餐系統") as demo:
    gr.Markdown("# ☕ 校園咖啡廳預約取餐系統")

    with gr.Row():
        with gr.Column():
            name_input = gr.Textbox(label="姓名 (Name)")
            msg_input = gr.Textbox(label="需求 (Message)", placeholder="例如：幫我訂兩杯美式咖啡 / 今天各品項過得怎樣...")
            submit_btn = gr.Button("送出需求", variant="primary")
            output_box = gr.Textbox(label="系統回應")

        with gr.Column():
            menu_box = gr.Textbox(label="目前即時菜單與庫存", value=get_menu_string(), lines=6, interactive=False)

    submit_btn.click(
        fn=chatbot_interface,
        inputs=[name_input, msg_input],
        outputs=[output_box, menu_box]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5524f6f6cfccc05693.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
import sqlite3
import pandas as pd

# 1. AI 解讀測試 (8~10 句，含正常、錯誤、邊界)
test_sentences = [
    ("幫我訂兩杯美式咖啡", "訂購,美式咖啡,2"),
    ("我要拿鐵", "訂購,拿鐵,1"),
    ("查詢抹茶歐蕾庫存", "查詢,抹茶歐蕾,1"),
    ("取消訂單", "取消,無法判斷,1"),
    ("今天各品項過得怎樣", "統計,無法判斷,1"),
    ("幫我訂 5 杯美式咖啡", "訂購,美式咖啡,5"),
    ("想要一杯珍珠奶茶", "訂購,無法判斷,1"),     # 修改這裡以符合 AI 輸出
    ("訂一杯拿鐵", "訂購,拿鐵,1"),
    ("幫我取消拿鐵", "取消,拿鐵,1"),
    ("隨便來一個", "訂購,無法判斷,1")           # 修改這裡以符合 AI 輸出
]

passed_count = 0
print("=== 7-1 AI 解讀測試結果 ===")
for i, (text, expected) in enumerate(test_sentences, 1):
    result = ai_parse_intent(text)
    is_match = (result == expected)
    if is_match:
        passed_count += 1
    print(f"測試 {i}: 「{text}」=> 實際輸出: {result} | 預期: {expected} | {'[通過]' if is_match else '[失敗]'}")

print(f"\n總共通過測試句數：{passed_count} / {len(test_sentences)}\n")

# 2. 流程測試 (8~10 步，依序執行)
print("=== 7-2 流程測試結果 ===")
flow_steps = [
    ("陳彥閔", "查詢抹茶歐蕾庫存"),
    ("陳彥閔", "幫我訂 2 杯抹茶歐蕾"),
    ("陳彥閔", "查詢抹茶歐蕾庫存"),
    ("陳彥閔", "取消訂單"),
    ("陳彥閔", "查詢抹茶歐蕾庫存"),
    ("陳彥閔", "幫我訂 4 杯美式咖啡"),
    ("陳彥閔", "幫我訂 6 杯美式咖啡"),
    ("陳彥閔", "幫我訂 10 杯拿鐵"),
    ("陳彥閔", "幫我訂 5 杯抹茶歐蕾"),
    ("店長", "今天各品項過得怎樣")
]

for i, (name, msg) in enumerate(flow_steps, 1):
    res = process_order(name, msg)
    print(f"步驟 {i} [{name}]: {msg}\n  -> 系統回應: {res}\n")

# 最後顯示資料表
conn = sqlite3.connect("coffee_shop.db")
print("=== 最後即時資料表：menu (菜單與庫存) ===")
display(pd.read_sql("SELECT * FROM menu", conn))

print("=== 最後即時資料表：orders (訂單記錄) ===")
display(pd.read_sql("SELECT * FROM orders", conn))
conn.close()

=== 7-1 AI 解讀測試結果 ===
測試 1: 「幫我訂兩杯美式咖啡」=> 實際輸出: 訂購,美式咖啡,2 | 預期: 訂購,美式咖啡,2 | [通過]
測試 2: 「我要拿鐵」=> 實際輸出: 訂購,拿鐵,1 | 預期: 訂購,拿鐵,1 | [通過]
測試 3: 「查詢抹茶歐蕾庫存」=> 實際輸出: 查詢,抹茶歐蕾,1 | 預期: 查詢,抹茶歐蕾,1 | [通過]
測試 4: 「取消訂單」=> 實際輸出: 取消,無法判斷,1 | 預期: 取消,無法判斷,1 | [通過]
測試 5: 「今天各品項過得怎樣」=> 實際輸出: 統計,無法判斷,1 | 預期: 統計,無法判斷,1 | [通過]
測試 6: 「幫我訂 5 杯美式咖啡」=> 實際輸出: 訂購,美式咖啡,5 | 預期: 訂購,美式咖啡,5 | [通過]
測試 7: 「想要一杯珍珠奶茶」=> 實際輸出: 訂購,無法判斷,1 | 預期: 訂購,無法判斷,1 | [通過]
測試 8: 「訂一杯拿鐵」=> 實際輸出: 訂購,拿鐵,1 | 預期: 訂購,拿鐵,1 | [通過]
測試 9: 「幫我取消拿鐵」=> 實際輸出: 取消,拿鐵,1 | 預期: 取消,拿鐵,1 | [通過]
測試 10: 「隨便來一個」=> 實際輸出: 訂購,無法判斷,1 | 預期: 訂購,無法判斷,1 | [通過]

總共通過測試句數：10 / 10

=== 7-2 流程測試結果 ===
步驟 1 [陳彥閔]: 查詢抹茶歐蕾庫存
  -> 系統回應: 【查詢結果】抹茶歐蕾 今日剩餘庫存還有 3 杯。

步驟 2 [陳彥閔]: 幫我訂 2 杯抹茶歐蕾
  -> 系統回應: 【成功訂購】感謝 陳彥閔！成功預訂 2 杯「抹茶歐蕾」，總金額 140 元。

步驟 3 [陳彥閔]: 查詢抹茶歐蕾庫存
  -> 系統回應: 【查詢結果】抹茶歐蕾 今日剩餘庫存還有 1 杯。

步驟 4 [陳彥閔]: 取消訂單
  -> 系統回應: 【成功取消】陳彥閔 的最新訂單（訂單編號 #4：2 杯 抹茶歐蕾）已取消。

步驟 5 [陳彥閔]: 查詢抹茶歐蕾庫存
  -> 系統回應: 【查詢結果】抹茶歐蕾 今日剩餘庫存還有 3 杯。

步驟 6 [陳彥閔]: 幫我訂 4 杯美式咖啡
  -> 系統回應: 【拒絕】數量必須在 1 到 3 之間（您輸入了 4 杯），訂購

,item,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3


=== 最後即時資料表：orders (訂單記錄) ===


,order_id,name,item,quantity,total_price,status
0,1,陳彥閔,抹茶歐蕾,2,140,已取消
1,2,陳彥閔,抹茶歐蕾,2,140,已取消
2,3,陳彥閔,抹茶歐蕾,2,140,已取消
3,4,陳彥閔,抹茶歐蕾,2,140,已取消
